# 🏪 Qianfan-OCR — Google Colab T4 (Step 3)

**Yêu cầu trên Google Drive (`MyDrive/PBL7/`):**
- `metadata.json` — danh sách video từ step 1
- `debug_frames/` — frames đã lọc từ step 2 (`debug_frames/{hashtag}/{video_id}/text_filtered/*.jpg`)

**Output:** `MyDrive/PBL7/data.json` — giữ full biến từ metadata (bỏ `frame`), thêm:
- `tenquan` — trích từ ảnh (VLM + ảnh)
- `diachi` — trích từ ảnh (VLM + ảnh)
- `mo_ta_rag` — VLM text-only viết 1 câu tự nhiên từ `tenquan` + `diachi` + suy luận từ hashtags

**Flow:** Mount Drive → Config → Cài lib → Load model → Hàm → Chạy OCR → Lưu

## Cell 1 — Mount Google Drive & Config đường dẫn

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

METADATA_PATH    = '/content/drive/MyDrive/PBL7/metadata.json'
DEBUG_FRAMES_DIR = '/content/drive/MyDrive/PBL7/debug_frames'
OUTPUT_JSON      = '/content/drive/MyDrive/PBL7/data.json'

for path, label in [(METADATA_PATH, 'metadata.json'), (DEBUG_FRAMES_DIR, 'debug_frames/')]:
    if os.path.exists(path):
        if os.path.isdir(path):
            from pathlib import Path as _P
            n = sum(1 for _ in _P(path).rglob('*'))
            print(f'✓ {label} ({n} files/folders)')
        else:
            size = os.path.getsize(path) / 1024 / 1024
            print(f'✓ {label} ({size:.1f} MB)')
    else:
        print(f'✗ KHÔNG TÌM THẤY: {path}')
        print('  → Kiểm tra lại cấu trúc thư mục trong Google Drive!')

Mounted at /content/drive
✓ metadata.json (3.8 MB)
✓ debug_frames/ (5005 files/folders)


## Cell 2 — Kiểm tra cấu trúc debug_frames

In [ ]:
from pathlib import Path

debug_frames_root = Path(DEBUG_FRAMES_DIR)
video_dirs = list(debug_frames_root.rglob('text_filtered'))
print(f'Tìm thấy {len(video_dirs)} thư mục text_filtered')

if video_dirs:
    print(f'Ví dụ: {video_dirs[0]}')
else:
    print('⚠ Không tìm thấy thư mục text_filtered — kiểm tra lại DEBUG_FRAMES_DIR trong Cell 1')

Tìm thấy 456 thư mục text_filtered
Ví dụ: /content/drive/MyDrive/PBL7/debug_frames/miquang/tk_7645978932830833940/text_filtered


## Cell 3 — Cài thư viện

In [ ]:
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q torch torchvision pillow accelerate
!pip install -q --upgrade huggingface_hub
import importlib, transformers
importlib.reload(transformers)
print(transformers.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.4/684.4 kB 20.7 MB/s eta 0:00:00
5.10.0.dev0


## Cell 4 — Load model (1 GPU Colab T4)

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import torch

# Dọn sạch VRAM trước khi load model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    for i in range(torch.cuda.device_count()):
        mem_free = torch.cuda.mem_get_info(i)[0] / 1e9
        mem_total = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'GPU {i}: {mem_free:.1f} / {mem_total:.1f} GB free')

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from PIL import Image

n_gpu = torch.cuda.device_count()
print(f'Số GPU: {n_gpu}')
for i in range(n_gpu):
    name = torch.cuda.get_device_name(i)
    mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {name} | {mem:.1f} GB')

MODEL_PATH = 'baidu/Qianfan-OCR'
print(f'\nLoading {MODEL_PATH}...')

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map='auto',
).eval()

processor = AutoProcessor.from_pretrained(MODEL_PATH)
print('✓ Model loaded!')
for i in range(n_gpu):
    alloc = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i} VRAM: {alloc:.1f} / {total:.1f} GB')

Số GPU: 1
  GPU 0: Tesla T4 | 15.6 GB

Loading baidu/Qianfan-OCR...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/68.4k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.6M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/26.1k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

✓ Model loaded!
  GPU 0 VRAM: 9.5 / 15.6 GB


## Cell 5 — Hàm xử lý

**Hàm chính:**
- `ocr_image(path)` — gọi VLM + ảnh → trích `tenquan` + `diachi` (1 GPU)
- `pick_best_tenquan(candidates, diachi, hashtag_chinh)` — VLM chọn tên hợp lý nhất từ mảng candidates
- `gen_mo_ta_rag(...)` — gọi VLM text-only → viết 1 câu từ `tenquan` + `diachi` + hashtags

In [ ]:
import json, re
from pathlib import Path

HASHTAG_TO_MON = {
    "banhbeo"       : "Bánh Bèo",
    "banhbotloc"    : "Bánh Bột Lọc",
    "banhcan"       : "Bánh Căn",
    "banhcanh"      : "Bánh Canh",
    "banhbao"       : "Bánh Bao",
    "banhcuon"      : "Bánh Cuốn",
    "banhkhot"      : "Bánh Khọt",
    "banhmi"        : "Bánh Mì",
    "banhtrangnuong": "Bánh Tráng Nướng",
    "banhxeo"       : "Bánh Xèo",
    "bunbohue"      : "Bún Bò Huế",
    "buncha"        : "Bún Chả",
    "bundaumamtom"  : "Bún Đậu Mắm Tôm",
    "bunmam"        : "Bún Mắm",
    "bunrieu"       : "Bún Riêu",
    "bunthitnuong"  : "Bún Thịt Nướng",
    "caolau"        : "Cao Lầu",
    "chaolong"      : "Cháo Lòng",
    "comtam"        : "Cơm Tấm",
    "goicuon"       : "Gỏi Cuốn",
    "hutieu"        : "Hủ Tiếu",
    "nemchua"       : "Nem Chua",
    "pho"           : "Phở",
    "xoi"           : "Xôi",
    "miquang"       : "Mì Quảng",
}

PROMPT_OCR = """Bạn là hệ thống OCR chuyên đọc biển hiệu cửa hàng Việt Nam.
Nhiệm vụ: Đọc toàn bộ chữ thực sự có trong ảnh, sau đó trích xuất 2 trường.

== TÊN QUÁN (tenquan) ==
- Gộp loại hình kinh doanh + tên thương hiệu nếu cả hai xuất hiện rõ trên biển hiệu.
- Đọc chính xác từng chữ cái, KHÔNG đoán mò hay thay thế tên.
- Khôi phục dấu tiếng Việt theo đúng tên thực tế.
- Không chắc chắn → null.

== ĐỊA CHỈ (diachi) ==
- Chỉ lấy khi có ĐỦ: số nhà + tên đường. Có thêm phường/quận/tỉnh càng tốt.
- Khôi phục dấu tiếng Việt dựa trên địa danh thực tế Việt Nam.
- Thiếu số nhà hoặc không chắc → null.

== QUY TẮC ==
1. Trả về JSON DUY NHẤT, không giải thích, không markdown, không code block.
2. KHÔNG bịa thông tin.
3. Bỏ qua: SĐT, giá tiền, giờ mở cửa, website, slogan, QR.

VÍ DỤ:
{"tenquan": "Bánh Bèo Bà Hường", "diachi": "126 Duy Tân, TP. Đà Nẵng"}
{"tenquan": "Phở Bò Anh Tuấn", "diachi": null}
{"tenquan": null, "diachi": "89 Trần Phú, Quận Hải Châu, TP. Đà Nẵng"}"""


def build_prompt_mo_ta_rag(tenquan, diachi, location_search, hashtag_chinh, hashtags_useful, hashtags_raw):
    tags_seen = set()
    tags_all  = []
    if hashtag_chinh:
        tags_all.append(f'#{hashtag_chinh}')
        tags_seen.add(hashtag_chinh)
    for t in (hashtags_useful or []) + (hashtags_raw or []):
        if t not in tags_seen:
            tags_all.append(t)
            tags_seen.add(t)
    tags_str = ' '.join(tags_all[:15])

    tenquan_str   = tenquan if tenquan else 'không rõ tên'
    diachi_str    = diachi  if diachi  else 'không rõ địa chỉ'
    location_str  = location_search if (location_search and location_search.strip().lower() != 'toàn quốc') else None
    location_line = f'- Khu vực: {location_str}\n' if location_str else ''

    return f"""Bạn đang viết mô tả ngắn cho một quán ăn Việt Nam để dùng trong hệ thống tìm kiếm RAG.

Thông tin có sẵn:
- Tên quán: {tenquan_str}
- Địa chỉ: {diachi_str}
{location_line}- Hashtags: {tags_str}

Nhiệm vụ: Viết ĐÚNG 1 câu tiếng Việt tự nhiên theo cấu trúc:
[mô tả loại quán/món dựa theo hashtag_chinh=#{hashtag_chinh or 'unknown'}], [tên quán], [địa chỉ], [đặc điểm thêm nếu có].
Ví dụ :
Tên quán: BÁNH KHỌT CÔ THU
Địa chỉ: Số 1 Bành Văn Trân, Q. Tân Bình
Quy tắc:
- Loại quán/món BẮT BUỘC suy từ hashtag_chinh (#{hashtag_chinh or 'unknown'}), KHÔNG dùng hashtag phụ để đoán loại quán
- KHÔNG bịa thông tin không có trong tên/địa chỉ/khu vực
- KHÔNG thêm giá tiền, giờ mở cửa, số điện thoại
- Chỉ trả về đúng 1 câu, không giải thích thêm

Ví dụ (hashtag_chinh=miquang):
Output: Quán mì Quảng đặc sản, Mì Quảng Bà Mua, 45 Lê Lợi, Đà Nẵng, hương vị truyền thống đậm đà.

Output:"""


def is_valid(value):
    return value is not None and str(value).strip().lower() not in ['null', 'none', '']


def is_valid_diachi(diachi):
    if not is_valid(diachi):
        return False
    if not re.search(r'\d', diachi):
        return False
    if len(diachi.strip()) < 8:
        return False
    return True


def _call_vlm(messages, max_new_tokens=256) -> str:
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(generated, skip_special_tokens=True)[0]


def parse_ocr_json(response: str) -> dict:
    try:
        clean = re.sub(r'```(?:json)?\n?', '', response).strip().strip('`').strip()
        match = re.search(r'\{.*?\}', clean, re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception:
        pass
    return {'tenquan': None, 'diachi': None}


def ocr_image(image_path: str) -> dict:
    try:
        print(f'    🖼 OCR ← {Path(image_path).name}')
        image    = Image.open(image_path).convert("RGB")
        messages = [{"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": PROMPT_OCR}
        ]}]
        response = _call_vlm(messages, max_new_tokens=256)
        result = parse_ocr_json(response)
        print(f'    🖼 → tenquan={result.get("tenquan")} | diachi={result.get("diachi")}')
        return result
    except Exception as e:
        torch.cuda.empty_cache()
        print(f'    ⚠ Lỗi {Path(image_path).name}: {e}')
        return {'tenquan': None, 'diachi': None}


def pick_best_tenquan(candidates: list, diachi: str, hashtag_chinh: str) -> str:
    """Dùng VLM chọn tên quán hợp lý nhất từ danh sách candidates.
    Nếu candidates rỗng → fallback 'Quán <tên món>'.
    """
    if not candidates:
        ten_mon = HASHTAG_TO_MON.get(hashtag_chinh, hashtag_chinh.capitalize())
        fallback = f'Quán {ten_mon}'
        print(f'  📝 tenquan_candidates rỗng → fallback: {fallback}')
        return fallback

    if len(candidates) == 1:
        return candidates[0]

    candidates_str = '\n'.join(f'- {c}' for c in candidates)
    prompt = f"""Bạn đang chọn tên quán ăn Việt Nam hợp lý nhất.

Địa chỉ quán: {diachi}
Các tên quán thu thập được từ nhiều frame khác nhau:
{candidates_str}

Hãy chọn tên quán chính xác và đầy đủ nhất (ưu tiên tên có đủ loại hình + thương hiệu).
Chỉ trả về đúng tên quán, không giải thích."""

    try:
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        response = _call_vlm(messages, max_new_tokens=64).strip()
        # Lấy dòng đầu tiên không rỗng
        for line in response.splitlines():
            line = line.strip().strip('- ').strip()
            if line:
                print(f'  📝 VLM chọn tenquan: {line} (từ {len(candidates)} candidates)')
                return line
    except Exception as e:
        torch.cuda.empty_cache()
        print(f'  ⚠ pick_best_tenquan lỗi: {e} → lấy candidate đầu tiên')

    return candidates[0]


def gen_mo_ta_rag(tenquan, diachi, location_search, hashtag_chinh, hashtags_useful, hashtags_raw) -> str:
    try:
        prompt   = build_prompt_mo_ta_rag(tenquan, diachi, location_search,
                                          hashtag_chinh, hashtags_useful, hashtags_raw)
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        response = _call_vlm(messages, max_new_tokens=128)
        for line in response.strip().splitlines():
            line = line.strip()
            line = re.sub(r'^[Oo]utput\s*:\s*', '', line).strip()
            if line:
                return line
        return response.strip()
    except Exception as e:
        torch.cuda.empty_cache()
        print(f'  ⚠ gen_mo_ta_rag lỗi: {e}')
        return ''


print('✓ Hàm OCR + pick_best_tenquan + gen_mo_ta_rag đã sẵn sàng!')

✓ Hàm OCR + pick_best_tenquan + gen_mo_ta_rag đã sẵn sàng!


## Cell 6 — Chạy OCR (1 GPU, duyệt hai đầu gặp nhau tuần tự)

**Logic mỗi video:**
1. Đọc `video_id` từ metadata → trỏ `debug_frames/{hashtag}/{video_id}/text_filtered/`
2. Duyệt frames theo thứ tự: `frame[0]` → `frame[n-1]` → `frame[1]` → `frame[n-2]` → ... (1 GPU, tuần tự)
3. Mỗi frame có `tenquan` hợp lệ → append vào `tenquan_candidates` (tiếp tục duyệt)
4. Khi tìm được `diachi` hợp lệ (có số nhà) → dừng, chốt `tenquan` qua VLM từ candidates
5. Nếu candidates rỗng → `tenquan = 'Quán ' + tên món từ hashtag`
6. Gọi `gen_mo_ta_rag()` → lưu record vào `data.json`

In [ ]:
import json, time
from pathlib import Path

debug_frames_root = Path(DEBUG_FRAMES_DIR)
output_path       = Path(OUTPUT_JSON)

with open(METADATA_PATH, 'r', encoding='utf-8') as f:
    all_records = json.load(f)
print(f'Tổng video trong metadata.json: {len(all_records)}')

if output_path.exists():
    with open(output_path, 'r', encoding='utf-8') as f:
        results = json.load(f)
    print(f'Resume: đã có {len(results)} record trong data.json')
else:
    results = []

done_video_ids = {r['video_id'] for r in results}

# Tìm mốc resume từ video cuối cùng trong data.json
if results:
    last_video_id = results[-1]['video_id']
    all_ids = [r['video_id'] for r in all_records]
    if last_video_id in all_ids:
        last_idx = all_ids.index(last_video_id)
        todo = [r for r in all_records[last_idx:] if r['video_id'] not in done_video_ids]
        print(f'Resume từ vị trí {last_idx} ({last_video_id}) trong metadata')
    else:
        todo = [r for r in all_records if r['video_id'] not in done_video_ids]
        print(f'Không tìm thấy video cuối trong metadata → fallback check done_video_ids')
else:
    todo = all_records
    print('Chạy mới từ đầu')

print(f'Cần xử lý: {len(todo)} video\n')

passed            = len(results)
skipped_no_dir    = 0
skipped_no_frame  = 0
skipped_no_result = 0
times             = []

for idx, rec in enumerate(todo, 1):
    video_id        = rec['video_id']
    hashtag         = rec.get('hashtag_chinh', 'unknown')
    location_search = rec.get('location_search', None)

    # ── Tìm thư mục frames ────────────────────────────────────────────────
    frame_dir = debug_frames_root / hashtag / video_id / 'text_filtered'
    if not frame_dir.exists():
        frame_dir_alt = debug_frames_root / hashtag / video_id
        if frame_dir_alt.exists():
            frame_dir = frame_dir_alt
        else:
            print(f'[{idx}/{len(todo)}] {video_id} → ✗ không có thư mục frames')
            skipped_no_dir += 1
            continue

    frames = sorted(
        list(frame_dir.rglob('*.jpg')) +
        list(frame_dir.rglob('*.jpeg')) +
        list(frame_dir.rglob('*.png'))
    )
    if not frames:
        print(f'[{idx}/{len(todo)}] {video_id} → ✗ thư mục rỗng')
        skipped_no_frame += 1
        continue

    loc_display = location_search if (location_search and location_search.strip().lower() != 'toàn quốc') else 'Toàn quốc'
    print(f'[{idx}/{len(todo)}] {video_id} | {len(frames)} frames | hashtag={hashtag} | khu vực={loc_display}')
    t_start = time.time()

    # ── Duyệt hai đầu gặp nhau, tuần tự 1 GPU ────────────────────────────
    tenquan_candidates = []
    best_diachi        = None
    checked            = 0

    left  = 0
    right = len(frames) - 1
    order = []
    while left <= right:
        order.append(frames[left])
        if right != left:
            order.append(frames[right])
        left  += 1
        right -= 1
    order = order[:7]

    for frame_idx, frame_path in enumerate(order):
        print(f'  Frame {frame_idx+1}/{len(order)}: {frame_path.name}')
        data = ocr_image(str(frame_path))
        checked += 1

        torch.cuda.empty_cache()

        tq = data.get('tenquan')
        dc = data.get('diachi')

        if is_valid(tq) and tq not in tenquan_candidates:
            tenquan_candidates.append(tq)
            print(f'  📝 Thêm tenquan candidate: {tq} (tổng: {len(tenquan_candidates)})')

        if best_diachi is None and is_valid_diachi(dc):
            best_diachi = dc
            print(f'  📍 Ghi nhận diachi: {dc}')
            print(f'  ✅ Có diachi → dừng duyệt, chốt tenquan')
            break
        elif is_valid(dc) and not is_valid_diachi(dc):
            print(f'  ⚠ Bỏ diachi → không có số nhà: {dc}')

    # ── Kết quả cuối ──────────────────────────────────────────────────────
    diachi = best_diachi

    if diachi is None:
        skipped_no_result += 1
        print(f'  ✗ Đã check {checked} frames | diachi=None → KHÔNG LƯU, bỏ qua')
        continue

    tenquan = pick_best_tenquan(tenquan_candidates, diachi, hashtag)

    location_search_clean = location_search if (location_search and location_search.strip().lower() != 'toàn quốc') else None

    mo_ta_rag = gen_mo_ta_rag(
        tenquan         = tenquan,
        diachi          = diachi,
        location_search = location_search_clean,
        hashtag_chinh   = rec.get('hashtag_chinh'),
        hashtags_useful = rec.get('hashtags_useful', []),
        hashtags_raw    = rec.get('hashtags_raw', [])
    )

    record = {k: v for k, v in rec.items() if k != 'frame'}
    record['tenquan']   = tenquan
    record['diachi']    = diachi
    record['mo_ta_rag'] = mo_ta_rag
    if location_search_clean:
        record['location_search'] = location_search_clean
    else:
        record.pop('location_search', None)

    results.append(record)
    passed += 1
    done_video_ids.add(video_id)

    elapsed = time.time() - t_start
    times.append(elapsed)
    avg = sum(times) / len(times)
    eta = avg * (len(todo) - idx)

    print(f'  ✓ tenquan      : {tenquan}')
    print(f'  ✓ diachi       : {diachi}')
    print(f'  ✓ location     : {location_search_clean}')
    print(f'  ✓ mo_ta_rag    : {mo_ta_rag}')
    print(f'  💾 ĐÃ LƯU vào data.json')
    print(f'  ⏱ {elapsed:.1f}s | TB: {avg:.1f}s/video | Còn lại: ~{eta/3600:.1f} giờ')

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    if idx % 10 == 0:
        print(f'  → Tiến độ: {idx}/{len(todo)} | Lưu: {passed} | Bỏ qua: {skipped_no_result}\n')

print('=' * 60)
print('TỔNG KẾT:')
print(f'  Tổng video xử lý    : {len(todo)}')
print(f'  Đã lưu (có diachi)  : {passed}')
print(f'  Không có thư mục    : {skipped_no_dir}')
print(f'  Thư mục rỗng        : {skipped_no_frame}')
print(f'  Không có diachi     : {skipped_no_result}')
print(f'  Thời gian TB/video  : {sum(times)/len(times):.1f}s' if times else '  Thời gian TB/video  : N/A')
print(f'  File output         : {output_path}')
print('=' * 60)

Tổng video trong metadata.json: 5593
Resume: đã có 71 record trong data.json
Resume từ vị trí 2270 (tk_7627756776745684242) trong metadata
Cần xử lý: 3322 video

[1/3322] tk_7628939972862643476 | 17 frames | hashtag=bunbohue | khu vực=Hồ Chí Minh
  Frame 1/7: frame_0001.jpg
    🖼 OCR ← frame_0001.jpg


[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


    🖼 → tenquan=None | diachi=None
  Frame 2/7: frame_0027.jpg
    🖼 OCR ← frame_0027.jpg


[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


    🖼 → tenquan=None | diachi=None
  Frame 3/7: frame_0004.jpg
    🖼 OCR ← frame_0004.jpg


...

    🖼 → tenquan=None | diachi=None
  ✗ Đã check 4 frames | diachi=None → KHÔNG LƯU, bỏ qua
[3322/3322] tk_7398363661833702672 | 2 frames | hashtag=miquang | khu vực=Toàn quốc
  Frame 1/2: frame_0010.jpg
    🖼 OCR ← frame_0010.jpg


[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


    🖼 → tenquan=None | diachi=None
  Frame 2/2: frame_0012.jpg
    🖼 OCR ← frame_0012.jpg


[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


    🖼 → tenquan=Quán ăn nhỏ Mi Quảng Quê Hương | diachi=476 Mã Lò, Bình Hưng Hòa A, Bình Tân, Hồ Chí Minh
  📝 Thêm tenquan candidate: Quán ăn nhỏ Mi Quảng Quê Hương (tổng: 1)
  📍 Ghi nhận diachi: 476 Mã Lò, Bình Hưng Hòa A, Bình Tân, Hồ Chí Minh
  ✅ Có diachi → dừng duyệt, chốt tenquan
  ✓ tenquan      : Quán ăn nhỏ Mi Quảng Quê Hương
  ✓ diachi       : 476 Mã Lò, Bình Hưng Hòa A, Bình Tân, Hồ Chí Minh
  ✓ location     : None
  ✓ mo_ta_rag    : Quán mì Quảng nổi tiếng, Mi Quảng Quê Hương, 476 Mã Lò, Bình Hưng Hòa A, Bình Tân, Hồ Chí Minh, đậm đà bản địa.
  💾 ĐÃ LƯU vào data.json
  ⏱ 39.0s | TB: 37.1s/video | Còn lại: ~0.0 giờ
TỔNG KẾT:
  Tổng video xử lý    : 3322
  Đã lưu (có diachi)  : 145
  Không có thư mục    : 3186
  Thư mục rỗng        : 0
  Không có diachi     : 62
  Thời gian TB/video  : 37.1s
  File output         : /content/drive/MyDrive/PBL7/data.json


## Cell 7 — Kiểm tra kết quả

In [ ]:
import json

with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
    data = json.load(f)

has_tenquan = sum(1 for r in data if r.get('tenquan'))
has_diachi  = sum(1 for r in data if r.get('diachi'))
has_rag     = sum(1 for r in data if r.get('mo_ta_rag'))

print(f'Tổng record   : {len(data)}')
print(f'Có tenquan    : {has_tenquan}')
print(f'Có diachi     : {has_diachi}')
print(f'Có mo_ta_rag  : {has_rag}')
print()
print('--- 2 record mẫu ---')
for r in data[:2]:
    print(json.dumps(r, ensure_ascii=False, indent=2))
    print()